In [2]:
import io
from pathlib import Path

import fitz
import pytesseract

from PIL import Image

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_text_splitters import RecursiveCharacterTextSplitter

c:\projects\document-intelligence-app\genenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

In [14]:
PDF_PATH = "./pdf_file/ml_digital_blank.pdf"

CHROMA_DIR = Path("./chroma_db")

COLLECTION_NAME = "scanned_pdf_rag"

EMBEDDING_MODEL = "sentence-transformers/multi-qa-MiniLM-L6-cos-v1"

GROQ_MODEL = "openai/gpt-oss-120b"

In [15]:
def load_pdf_with_ocr(pdf_path: str) -> list[Document]:
    """
    Read a scanned PDF page by page and convert each page
    into a LangChain Document using OCR.
    """

    pdf = fitz.open(pdf_path)

    documents = []

    print(f"Total pages: {len(pdf)}")

    for page_number, page in enumerate(pdf):

        print(f"Processing page {page_number + 1}/{len(pdf)}...")

        # ----------------------------------------------------
        # Convert PDF page into image
        # ----------------------------------------------------

        pix = page.get_pixmap(
            dpi=300
        )

        # ----------------------------------------------------
        # Convert image bytes -> PIL Image
        # ----------------------------------------------------

        image = Image.open( io.BytesIO( pix.tobytes("png") ) )
            
        # ----------------------------------------------------
        # OCR
        # ----------------------------------------------------

        text = pytesseract.image_to_string(
            image
        )

        text = text.strip()

        # ----------------------------------------------------
        # Skip empty pages
        # ----------------------------------------------------

        if not text:
            print(
                f"WARNING: Page {page_number + 1} "
                f"contains no extractable text."
            )
            continue

        # ----------------------------------------------------
        # Create LangChain Document
        # ----------------------------------------------------

        document = Document(
            page_content=text,

            metadata={
                "source": str(pdf_path),
                "page": page_number + 1,
                "loader": "tesseract_ocr"
            }
        )

        documents.append(document)

    pdf.close()

    print(
        f"\nSuccessfully created "
        f"{len(documents)} documents."
    )

    return documents

In [16]:
documents = load_pdf_with_ocr(
    PDF_PATH
)

print("\nFirst document:")
print(documents[0])


Total pages: 5
Processing page 1/5...
Processing page 2/5...
Processing page 3/5...
Processing page 4/5...
Processing page 5/5...

Successfully created 3 documents.

First document:
page_content='Introduction to Machine Learning

1. What Is Machine Learning?

Machine learning is a branch of artificial intelligence that enables computer systems to learn patterns
from data and use those patterns to make predictions or decisions. Instead of explicitly programming
every rule, we provide examples and allow an algorithm to estimate a useful relationship between
inputs and outputs.

2. Types of Machine Learning

Supervised learning uses labeled examples. Common tasks include classification and regression.
For example, a model can learn from historical house data to predict the price of a new house.

Unsupervised learning works with data without target labels. Clustering customers according to
purchasing behavior is a common example.

Reinforcement learning learns through interaction with an e

In [17]:
splitter = RecursiveCharacterTextSplitter(

    chunk_size=800,

    chunk_overlap=100,

    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = splitter.split_documents(
    documents
)

print("\nNumber of chunks:", len(chunks))

print("\nFirst chunk:")
print(chunks[0].page_content)


Number of chunks: 6

First chunk:
Introduction to Machine Learning

1. What Is Machine Learning?

Machine learning is a branch of artificial intelligence that enables computer systems to learn patterns
from data and use those patterns to make predictions or decisions. Instead of explicitly programming
every rule, we provide examples and allow an algorithm to estimate a useful relationship between
inputs and outputs.

2. Types of Machine Learning

Supervised learning uses labeled examples. Common tasks include classification and regression.
For example, a model can learn from historical house data to predict the price of a new house.

Unsupervised learning works with data without target labels. Clustering customers according to
purchasing behavior is a common example.


In [18]:
embeddings = HuggingFaceEmbeddings(

    model_name=EMBEDDING_MODEL
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 631.74it/s]


In [19]:
vector_store = Chroma(

    collection_name=COLLECTION_NAME,

    embedding_function=embeddings,

    persist_directory=str(CHROMA_DIR)
)

In [20]:
vector_store.get()

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

In [21]:
vector_store.add_documents(
    chunks
)

print("\nDocuments successfully added to ChromaDB.")


Documents successfully added to ChromaDB.


In [23]:
data = vector_store.get(include=["documents","metadatas"])

In [47]:
data

{'ids': ['330bc57e-f0db-4838-9854-f9ce281a6ba7',
  '46bcd895-a02b-4423-92f2-298171ec8454',
  'c2c180b5-4d34-4487-8818-4014a39ad4b9',
  '71917fe0-b952-4e01-ae9b-8051fd2256ac',
  '430a003a-e138-4083-bd45-b086e39ee9ed',
  '501aef8f-ed94-496d-8486-9ca56b543aae'],
 'embeddings': None,
 'documents': ['Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers

In [32]:
docs = [Document(page_content= doc, metadata=meta or {}) for doc , meta in zip(data['documents'],data['metadatas'])]
docs


[Document(metadata={'source': './pdf_file/ml_digital_blank.pdf', 'page': 1, 'loader': 'tesseract_ocr'}, page_content='Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers according to\npurchasing behavior is a common example.'),
 Document(metadata={'loader': 'tesseract_ocr', 'page': 1, 'source': './pdf_file/ml_digital_blank.pdf'}, page_content='Re

In [26]:
data['metadatas']

[{'source': './pdf_file/ml_digital_blank.pdf',
  'page': 1,
  'loader': 'tesseract_ocr'},
 {'loader': 'tesseract_ocr',
  'page': 1,
  'source': './pdf_file/ml_digital_blank.pdf'},
 {'page': 2,
  'loader': 'tesseract_ocr',
  'source': './pdf_file/ml_digital_blank.pdf'},
 {'source': './pdf_file/ml_digital_blank.pdf',
  'page': 2,
  'loader': 'tesseract_ocr'},
 {'loader': 'tesseract_ocr',
  'source': './pdf_file/ml_digital_blank.pdf',
  'page': 4},
 {'source': './pdf_file/ml_digital_blank.pdf',
  'loader': 'tesseract_ocr',
  'page': 4}]

In [29]:
res = vector_store.similarity_search(query="what is machine learning", k=2)
for r in res:
    print(r.page_content)

Introduction to Machine Learning

1. What Is Machine Learning?

Machine learning is a branch of artificial intelligence that enables computer systems to learn patterns
from data and use those patterns to make predictions or decisions. Instead of explicitly programming
every rule, we provide examples and allow an algorithm to estimate a useful relationship between
inputs and outputs.

2. Types of Machine Learning

Supervised learning uses labeled examples. Common tasks include classification and regression.
For example, a model can learn from historical house data to predict the price of a new house.

Unsupervised learning works with data without target labels. Clustering customers according to
purchasing behavior is a common example.
The Machine Learning Workflow

3. From Data to Model

A typical machine learning project begins with data collection. The data is then cleaned, explored,
transformed, and divided into training and evaluation sets. Features are selected or constructed to
re

### BM25

In [41]:
from rank_bm25 import BM25Okapi

class BM25Index:
    def __init__(self):
        self.documents = []
        self.model = None

    def build(self, documents):
        self.documents = documents

        tokenized = [
           doc.page_content.lower().split() 
           for doc in documents
        ]
        self.model = BM25Okapi(tokenized) if tokenized else None
        return self.model

    def search(self, query: str, k: int = 20):
        if not self.model :
            return []
        scores = self.model.get_scores(query.lower().split())
        indices = sorted(range(len(scores)), key= lambda i: scores[i], reverse=True)

        return [(self.documents[i], float(scores[i])) for i in indices]
        
_bm25= BM25Index()



In [42]:
docs

[Document(metadata={'source': './pdf_file/ml_digital_blank.pdf', 'page': 1, 'loader': 'tesseract_ocr'}, page_content='Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers according to\npurchasing behavior is a common example.'),
 Document(metadata={'loader': 'tesseract_ocr', 'page': 1, 'source': './pdf_file/ml_digital_blank.pdf'}, page_content='Re

In [43]:
model = _bm25.build(docs)
model


In [44]:
query = "what is machine learning"
keyword = _bm25.search(query, k =5)

In [46]:
keyword[0]

(Document(metadata={'source': './pdf_file/ml_digital_blank.pdf', 'page': 1, 'loader': 'tesseract_ocr'}, page_content='Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers according to\npurchasing behavior is a common example.'),
 1.4999239575778254)

In [34]:
len(chunks)

6

In [40]:
from rank_bm25 import BM25Okapi

tokenized = [
           doc.page_content.lower().split() 
           for doc in chunks
        ]
print(len(tokenized))
tokenized[0][:50]


6


['introduction',
 'to',
 'machine',
 'learning',
 '1.',
 'what',
 'is',
 'machine',
 'learning?',
 'machine',
 'learning',
 'is',
 'a',
 'branch',
 'of',
 'artificial',
 'intelligence',
 'that',
 'enables',
 'computer',
 'systems',
 'to',
 'learn',
 'patterns',
 'from',
 'data',
 'and',
 'use',
 'those',
 'patterns',
 'to',
 'make',
 'predictions',
 'or',
 'decisions.',
 'instead',
 'of',
 'explicitly',
 'programming',
 'every',
 'rule,',
 'we',
 'provide',
 'examples',
 'and',
 'allow',
 'an',
 'algorithm',
 'to',
 'estimate']

In [45]:
model = BM25Okapi(tokenized)
model

In [46]:
query = " what is machine learning"
scores = model.get_scores(query.lower().split())
scores

array([1.49992396, 0.37785519, 0.36345418, 0.        , 0.25825273,
       0.24729409])

In [54]:
indices = sorted(range(len(scores)),key = lambda i :scores[i], reverse=True)[:2]
indices


[0, 1]

In [ ]:
a =0

In [1]:
lst = (10,2,5,1,6)  # 1,2,5,6,10

len(lst)



5

In [5]:
scores = sorted(lst, reverse=True)
scores

[10, 6, 5, 2, 1]

In [10]:
indices = sorted(range(len(lst)), key =lambda i : lst[i], reverse=True)
print(indices)
res = [lst[i] for i in indices]
res

[0, 4, 2, 1, 3]


[10, 6, 5, 2, 1]

In [56]:
res = [ (chunks[i].page_content, float(scores[i]))  for i in indices]
res

[('Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers according to\npurchasing behavior is a common example.',
  1.4999239575778254),
 ('Reinforcement learning learns through interaction with an environment. An agent receives rewards\nor penalties and gradually learns a policy for choosing actions.',
  0.3778551909502029)]

In [30]:
chunks

[Document(metadata={'source': './pdf_file/ml_digital_blank.pdf', 'page': 1, 'loader': 'tesseract_ocr'}, page_content='Introduction to Machine Learning\n\n1. What Is Machine Learning?\n\nMachine learning is a branch of artificial intelligence that enables computer systems to learn patterns\nfrom data and use those patterns to make predictions or decisions. Instead of explicitly programming\nevery rule, we provide examples and allow an algorithm to estimate a useful relationship between\ninputs and outputs.\n\n2. Types of Machine Learning\n\nSupervised learning uses labeled examples. Common tasks include classification and regression.\nFor example, a model can learn from historical house data to predict the price of a new house.\n\nUnsupervised learning works with data without target labels. Clustering customers according to\npurchasing behavior is a common example.'),
 Document(metadata={'source': './pdf_file/ml_digital_blank.pdf', 'page': 1, 'loader': 'tesseract_ocr'}, page_content='Re

In [ ]:
llm = ChatGroq(

    model=GROQ_MODEL,

    temperature=0
)


In [ ]:
prompt = ChatPromptTemplate.from_messages([

    (
        "system",
        """
You are a helpful question-answering assistant.

Answer the user's question using ONLY the
provided context.

Rules:

1. Do not use outside knowledge.
2. If the answer is not present in the context,
   say:

   "I don't have enough information in the
   provided document."

3. Do not invent information.
4. Give a concise but complete answer.
5. If possible, mention the page number
   where the information was found.

Context:

{context}
"""
    ),

    (
        "human",
        "{question}"
    )
])


In [51]:
from collections import defaultdict , deque 

_memory = defaultdict( lambda : deque(maxlen=3))
print(_memory)

defaultdict(<function <lambda> at 0x000001D96A292B60>, {})


In [57]:
m = deque(maxlen=3)
m.append("msg1")
m.append("msg2")
m.append("msg3")

In [61]:
m.append("msg5")

In [62]:
m

deque(['msg3', 'msg4', 'msg5'], maxlen=3)

In [66]:
memory = {}


In [68]:
if "session_1" not in memory:
    memory["session_1"] = []

memory["session_1"].append("Hello")
memory["session_1"].append("what is machine learning")
memory

{'session_1': ['Hello', 'Hello', 'what is machine learning']}

In [ ]:
import sqlite3 
import os
conn = sqlite3.connect("data/my_db.db")
conn.close()

In [4]:
import uuid 

ids = str(uuid.uuid4())
ids

'b921e337-3ac5-449d-b2e6-a1361c73c334'

In [5]:
from datetime import datetime 

datetime.utcnow().isoformat()

'2026-10-08T04:15:25.664756'

In [1]:
import pymupdf
import hashlib
with open("pdf_file/lti.pdf", "rb") as pdf :
     data = pdf.read()
     digest = hashlib.sha256(data).hexdigest()
print(digest)    

95c57d4f21efb08fd212edc650dd3c6fc58b20071d7c9c6364b4b11e30d075a5


In [2]:

history = " i am here"
question = " where are you"
print(f"Conversation context:\n{history}\n\nCurrent question:\n{question}")

Conversation context:
 i am here

Current question:
 where are you


In [4]:
from collections import defaultdict, deque


_memory = defaultdict(lambda: deque(maxlen=5))
_memory

defaultdict(<function __main__.<lambda>()>, {})

In [5]:
_memory.items()

dict_items([])

In [12]:
_memory['101'].append(7)
_memory

defaultdict(<function __main__.<lambda>()>,
            {'101': deque([3, 4, 5, 6, 7], maxlen=5)})

In [2]:
items={'role':  "user","content" :"what is machine learning"}


print(f"{items['role']}:{items['content']}")

user:what is machine learning


In [11]:
%pip install openpyxl

  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
Using cached et_xmlfile-2.0.0-py3-none-any.whl (18 kB)
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd 
import numpy as np 
import openpyxl

In [13]:
print(openpyxl.__version__)

3.1.5


In [15]:
data = pd.read_excel("movies_raw.xlsx", sheet_name="movies")
data.head()

,movie_id,title,industry,release_year,imdb_rating,studio,language_id
0,101,K.G.F: Chapter 2,Bollywood,2022,8.4,Hombale Films,3
1,102,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,5
2,103,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,5
3,104,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,5
4,105,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,5


In [ ]:
data['release_year']

np.int64(1946)

In [17]:
min(data['release_year'])

1946

In [19]:
# Print all the years where more than 2 movies were released
data.head()

,movie_id,title,industry,release_year,imdb_rating,studio,language_id
0,101,K.G.F: Chapter 2,Bollywood,2022,8.4,Hombale Films,3
1,102,Doctor Strange in the Multiverse of Madness,Hollywood,2022,7.0,Marvel Studios,5
2,103,Thor: The Dark World,Hollywood,2013,6.8,Marvel Studios,5
3,104,Thor: Ragnarok,Hollywood,2017,7.9,Marvel Studios,5
4,105,Thor: Love and Thunder,Hollywood,2022,6.8,Marvel Studios,5


In [25]:
# Count movies released in each year
year_counts = data.groupby("release_year").size()
year_counts

release_year
1946    1
1955    1
1972    1
1975    1
1993    2
1994    1
1995    1
1997    1
2000    1
2001    1
2003    1
2006    1
2007    1
2008    1
2009    2
2010    1
2011    1
2013    1
2014    3
2015    3
2017    1
2018    3
2019    2
2021    2
2022    5
dtype: int64

In [26]:
year_counts[year_counts>2]

release_year
2014    3
2015    3
2018    3
2022    5
dtype: int64

In [37]:
df = data.groupby("release_year").agg(
    count=("title", "size"), title=("title", lambda x: " || ".join(x))
).reset_index()
df

,release_year,count,title
0,1946,1,It's a Wonderful Life
1,1955,1,Pather Panchali
2,1972,1,The Godfather
3,1975,1,Sholay
4,1993,2,Schindler's List || Jurassic Park
5,1994,1,The Shawshank Redemption
6,1995,1,Dilwale Dulhania Le Jayenge
7,1997,1,Titanic
8,2000,1,Gladiator
9,2001,1,Kabhi Khushi Kabhie Gham


In [38]:
df[df['count']>2]

,release_year,count,title
18,2014,3,Interstellar || PK || Captain America: The Win...
19,2015,3,Bajirao Mastani || Baahubali: The Beginning |...
21,2018,3,Avengers: Infinity War || Sanju || Race 3
24,2022,5,K.G.F: Chapter 2 || Doctor Strange in the Mult...
